<a href="https://colab.research.google.com/github/pop123-ux/HuggingFace-Project-Learning/blob/main/GRPO_Implementation_in_PyTorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

First of all, we load a model and generate multiple responses for a given question:

In [5]:
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2-Math-1.5B"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Input prompt
prompt = "Solve y = 3x**2 + 2x + 5 for x = 3, y = "
inputs = tokenizer(prompt, return_tensors="pt", padding=True)
input_ids = inputs['input_ids'].to(device) # Shape (1, prompt_len)
attention_mask = inputs['attention_mask'].to(device)

# Step 1: Generate 8 responses (B = 2 groups, G = 4 responses per group)
batch_size, num_generations = 2, 4
outputs = model.generate(
    input_ids=input_ids, # Shape: (1, prompt_len)
    attention_mask = attention_mask,
    max_new_tokens=1, #seq_len = 1 (single token per response)
    num_return_sequences=batch_size * num_generations, # 8 responses total
    do_sample=True,
    top_k=10,
    temperature=0.6,
    pad_token_id=tokenizer.eos_token_id,
    return_dict_in_generate=True,
    output_scores=True,
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [4]:
#outputs

## Calculating Rewards ##

Now, we need to determine which responses are correct and assign rewards accordingly:

With GRPO, with the same sample prompt, we generate multiple completions. So, for instance for our prompts of "Solve y = 3x**2 + 2x + 5 for x = 3, y = " and Solve y = 3x**2 + 2x +5 for x = 1, y =" we have two group of generated outputs for the given prompt one is say

- [5, 6, 38, 10] and the other is
- [10, 38, 2, 16] while the correct answer is 38 and 10.

In practice these reward scores are achieved by a rule-based reward function that assigns rewards based on the correctness of the response or a more complex neural network-based model that can be trained to assign rewards based on the correctness of the response or a mixed of both. But for the sake of simplicity let's say our reward per response is 1 if the response is correct and 0 if it is wrong, therefore:

## reward_1 = [0, 0, 1, 1] ##
## reward_2 = [1, 1, 0, 0] ##

next we get the group_wise mean and std of the rewards:

In [6]:
rewards = torch.tensor([0, 0, 1, 1, 1, 1, 0, 0], dtype=torch.float32)
num_generations = 4

# Group rewards: Shape (B, G) = 2, 4
rewards_grouped = rewards.view(-1, num_generations)

# Mean per group: Shape (B,) = (2,)
mean_grouped_rewards = rewards_grouped.mean(dim=1)

# Std per group: Shape (B,) = (2,)
std_grouped_rewards = rewards_grouped.std(dim=1)

# Broadcast to match rewards and normalize: Shape (B * G,) = (8,)
# why do we need to broadcast you may ask, because we need to calculate the advantage values for each response within the group
mean_grouped_rewards_broadcasted = mean_grouped_rewards.repeat_interleave(num_generations, dim=0)
std_grouped_rewards_broadcasted = std_grouped_rewards.repeat_interleave(num_generations, dim=0)

In [7]:
print(f"Grouped Rewards: {rewards_grouped}")
print(f"Mean per group: {mean_grouped_rewards}")
print(f"Std per group: {std_grouped_rewards}")
print(f"Broadcasted Mean: {mean_grouped_rewards_broadcasted}")
print(f"Broadcasted Std: {std_grouped_rewards_broadcasted}")

Grouped Rewards: tensor([[0., 0., 1., 1.],
        [1., 1., 0., 0.]])
Mean per group: tensor([0.5000, 0.5000])
Std per group: tensor([0.5774, 0.5774])
Broadcasted Mean: tensor([0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000])
Broadcasted Std: tensor([0.5774, 0.5774, 0.5774, 0.5774, 0.5774, 0.5774, 0.5774, 0.5774])


Now we can calculate the advantage values for each response:

In [9]:
advantages = (rewards - mean_grouped_rewards_broadcasted) / (std_grouped_rewards_broadcasted + 1e-8)

This will output:

In [10]:
print(f"Advantages: {advantages}")

Advantages: tensor([-0.8660, -0.8660,  0.8660,  0.8660,  0.8660,  0.8660, -0.8660, -0.8660])


For reward_1 = [0, 0, 1, 1]

1 - 0.5 / 0.5774 ~ 0.8660
0 - 0.5 / 0.5774 ~ -0.8660

For reward_2 = [1, 1, 0, 0]: Same pattern.

In practice, we need to have the shape of (B, G) = (2, 4) to match the logits shape, therefore we need to unsqueeze the advantages tensor to have the shape of (B*G, 1) = (8, 1) to match the logits shape.

In [11]:
# Shape (B * G, 1) = (8, 1) to match the logits shape
advantages = advantages.unsqueeze(1)

To compute the probability ratio between the new and old policies, we first need the log probabilities of the generated tokens under both policies.

**`per_token_logps` (Old Policy Log Probabilities):** These are obtained from the `outputs.scores` attribute of the `model.generate` call, which provides the logits for the generated tokens under the policy that generated them.

**`new_per_token_logps` (Current Policy Log Probabilities):** These are obtained by performing a new forward pass of the *generated sequences* through the model and extracting the log probabilities for the *generated tokens* from the resulting logits. This evaluates the generated tokens under the current model's parameters.

In [17]:
print(f"per_token_logps shape: {per_token_logps.shape}")
print(f"new_per_token_logps shape: {new_per_token_logps.shape}")

per_token_logps shape: torch.Size([8])
new_per_token_logps shape: torch.Size([8])


which will output:

In [12]:
print(f"Advantages: {advantages}")

Advantages: tensor([[-0.8660],
        [-0.8660],
        [ 0.8660],
        [ 0.8660],
        [ 0.8660],
        [ 0.8660],
        [-0.8660],
        [-0.8660]])


now we are good, let's move to the next step of updating the policy based on the advantage values.

## 3. Updating the Policy ##

Finally, we use the advantage values to update our model:

In [15]:
# 1. Calculate per_token_logps (Log probabilities of generated tokens under the policy that generated them)
# outputs.scores[0] contains the logits for the first (and only) generated token for each sequence.
logits = outputs.scores[0] # Shape (batch_size * num_generations, vocab_size)

# The generated tokens are the last `max_new_tokens` (which is 1) in each sequence.
# outputs.sequences has shape (batch_size * num_generations, prompt_len + max_new_tokens)
generated_tokens = outputs.sequences[:, -1] # Shape (batch_size * num_generations,)

# Apply log_softmax to get log probabilities
log_probs = F.log_softmax(logits, dim=-1)

# Gather the log probabilities corresponding to the actually generated token IDs
per_token_logps = log_probs.gather(dim=-1, index=generated_tokens.unsqueeze(-1)).squeeze(-1)


# 2. Calculate new_per_token_logps (Log probabilities of generated tokens under the current policy)
# Perform a forward pass of the *entire generated sequences* through the model
# and get the logits for the generated token position.

# Create attention mask for the full sequences
full_attention_mask = (outputs.sequences != tokenizer.pad_token_id).long()

forward_pass_outputs = model(
    input_ids=outputs.sequences,
    attention_mask=full_attention_mask
)

# Get the logits for the generated token position (the last token in each sequence)
# The logits tensor has shape (batch_size * num_generations, sequence_length, vocab_size)
new_logits_for_generated_token = forward_pass_outputs.logits[:, -1, :] # Shape (batch_size * num_generations, vocab_size)

# Apply log_softmax to get new log probabilities
new_log_probs = F.log_softmax(new_logits_for_generated_token, dim=-1)

# Gather the log probabilities corresponding to the actually generated token IDs
new_per_token_logps = new_log_probs.gather(dim=-1, index=generated_tokens.unsqueeze(-1)).squeeze(-1)

In [18]:
# Compute probability ratio between new and old policies
ratio = torch.exp(
    new_per_token_logps - per_token_logps
)

In [19]:
print(f"Ratio: {ratio}")

Ratio: tensor([0.1926, 0.2082, 0.1373, 0.1926, 0.1373, 0.3125, 0.1373, 0.3125],
       grad_fn=<ExpBackward0>)


Torch clamp understanding:

In [22]:
import torch

# Example tensor
x = torch.tensor([0.5, 1.0, 1.5, 2.0])

# Clamp values between 0.8 and 1.2
clamped_x = torch.clamp(x, min=0.8, max=1.2)

print(f"Original: {x}")
print(f"Clamped:  {clamped_x}")

Original: tensor([0.5000, 1.0000, 1.5000, 2.0000])
Clamped:  tensor([0.8000, 1.0000, 1.2000, 1.2000])


We note that the per_token_logps can be achieved by passing the generated outputs to the model and get the logits and then apply the softmax function to get the probabilities F.softmax(logits, dim=-1).

In [21]:
# Clipping Function
eps = 0.2 # e.g. 0.2
pg_losses1 = -advantages * ratio # Shape: (B*G, seq_len)
pg_losses2 = -advantages * torch.clamp(
    ratio, 1.0 - eps, 1.0 + eps
)
pg_loss_max = torch.max(pg_losses1, pg_losses2)

# Calculate KL divergence approximation
# Formula: kl = exp(log_ratio) - log_ratio - 1
log_ratio = new_per_token_logps - per_token_logps
per_token_kl = torch.exp(log_ratio) - log_ratio - 1

# Now combine with KL penalty
per_token_loss = pg_loss_max + 0.2 * per_token_kl

per_token_kl can also be calculated as follows:

In [24]:
per_token_kl = F.kl_div(
    F.log_softmax(new_per_token_logps, dim=-1),
    F.softmax(per_token_logps, dim=-1),
    reduction='none',
).sum(dim=-1, keepdim=True)
per_token_kl

tensor([0.0569], grad_fn=<SumBackward1>)

## Summary: ##

- GRPO compares multiple outputs within a group to determine which ones are better than others, without requiring a separate value model.
- The advantage calculation standardizes rewards to identify which responses are above or below average.
- The policy update uses a clipped objective function with a KL divergence penalty to ensure stable learning.

This approach is particularly powerful for mathematical reasoning tasks, where correctness can be objectively verified. The GRPO method allows for more efficient training compared to traditional RLHF approaches that require a separate critic model.

We may consider experimenting with different group sizes, reward functions, and KL penalty coefficients to see how they affect our model's performance.